# Backtesting with CBSC Strategy SDK

This notebook demonstrates how to use the `BacktestAdapter` to run strategy backtests, analyze results, and optimize parameters.

## Setup

First, import the required modules and initialize the adapter.

In [ ]:
import asyncio
from datetime import date, datetime
from cbsc_strategy_sdk import BacktestAdapter, ParameterOptimizer

print("CBSC Strategy SDK imported successfully")

## Running a Simple Backtest

Let's run a basic backtest with a simple RSI strategy.

In [ ]:
async def run_simple_backtest():
    async with BacktestAdapter(api_base="http://localhost:3003") as adapter:
        # Define backtest parameters
        result = await adapter.run_backtest(
            strategy_code="rsi_mean_reversion",
            symbols=["AAPL", "0700.HK"],
            start_date=date(2024, 1, 1),
            end_date=date(2024, 12, 31),
            parameters={
                "rsi_period": 14,
                "oversold_threshold": 30,
                "overbought_threshold": 70,
            },
            initial_capital=1000000,
            commission_rate=0.001,
        )
        
        return result

# Run the backtest
result = await run_simple_backtest()

## Viewing Results

The backtest result provides several methods for analyzing performance.

In [ ]:
# Print a text summary
print(result.summary())

## Equity Curve Visualization

Plot the equity curve with drawdown overlay.

In [ ]:
# Plot equity curve with drawdown
fig = result.plot_equity_curve(figsize=(12, 8), show_drawdown=True)
fig.show()

## Trade Analysis

Convert trades to a pandas DataFrame for detailed analysis.

In [ ]:
import pandas as pd

# Get trades as DataFrame
trades_df = result.to_dataframe()

# Display summary statistics
print(f"Total Trades: {len(trades_df)}")
print(f"Win Rate: {trades_df['is_winner'].mean() * 100:.1f}%")
print(f"Average PnL: ${trades_df['pnl'].mean():.2f}")
print(f"Average Holding Period: {trades_df['holding_period_days'].mean():.1f} days")

# Show best and worst trades
print("\nBest 5 Trades:")
print(trades_df.nlargest(5, 'pnl')[['symbol', 'entry_time', 'exit_time', 'pnl', 'pnl_percent']])

print("\nWorst 5 Trades:")
print(trades_df.nsmallest(5, 'pnl')[['symbol', 'entry_time', 'exit_time', 'pnl', 'pnl_percent']])

## Monthly Returns Heatmap

Visualize monthly returns to identify seasonality patterns.

In [ ]:
# Plot monthly returns heatmap
fig = result.plot_monthly_returns(figsize=(12, 6))
fig.show()

## Progress Tracking

Track backtest progress with callbacks.

In [ ]:
from cbsc_strategy_sdk import BacktestProgress

async def run_with_progress():
    # Create progress tracker
    progress = BacktestProgress(total_steps=100)
    
    # Add callback for logging
    progress.add_callback(lambda pct, msg: print(f"Progress: {pct:.1f}% - {msg}"))
    
    async with BacktestAdapter() as adapter:
        result = await adapter.run_backtest(
            strategy_code="rsi_mean_reversion",
            symbols=["AAPL"],
            start_date=date(2024, 1, 1),
            end_date=date(2024, 12, 31),
            parameters={"rsi_period": 14},
            on_progress=lambda pct: print(f"\rBacktest progress: {pct:.1f}%", end=""),
        )
    
    print("\nBacktest complete!")
    return result

# Run with progress
result = await run_with_progress()

## Parameter Optimization

Use grid search to find optimal parameters.

In [ ]:
async def optimize_parameters():
    async with BacktestAdapter() as adapter:
        optimizer = ParameterOptimizer(adapter)
        
        # Define parameter grid
        parameter_grid = {
            "rsi_period": [10, 14, 20, 25],
            "oversold_threshold": [20, 25, 30],
            "overbought_threshold": [70, 75, 80],
        }
        
        # Run grid search
        result = await optimizer.grid_search(
            strategy_code="rsi_mean_reversion",
            symbols=["AAPL"],
            start_date=date(2024, 1, 1),
            end_date=date(2024, 6, 30),  # Shorter period for faster optimization
            parameter_grid=parameter_grid,
            metric="sharpe_ratio",
            on_progress=lambda pct: print(f"Optimization progress: {pct:.1f}%"),
        )
        
        return result

# Run optimization
opt_result = await optimize_parameters()

# Display results
print(opt_result.summary())

## Optimization Results Analysis

Convert optimization results to DataFrame for analysis.

In [ ]:
# Get results as DataFrame
results_df = opt_result.to_dataframe()

# Display top 5 parameter combinations
print("Top 5 Parameter Combinations (by Sharpe Ratio):")
print(results_df.nlargest(5, 'sharpe_ratio')[['rsi_period', 'oversold_threshold', 'overbought_threshold', 'sharpe_ratio', 'total_return', 'max_drawdown']])

# Analyze parameter sensitivity
print("\nParameter Sensitivity Analysis:")
for param in ['rsi_period', 'oversold_threshold', 'overbought_threshold']:
    if param in results_df.columns:
        avg_sharpe = results_df.groupby(param)['sharpe_ratio'].mean().sort_values(ascending=False)
        print(f"\n{param}:")
        print(avg_sharpe)

## Random Search

Use random search for continuous parameter spaces.

In [ ]:
async def random_search_optimization():
    async with BacktestAdapter() as adapter:
        optimizer = ParameterOptimizer(adapter)
        
        # Define parameter bounds
        parameter_bounds = {
            "rsi_period": (10, 25),
            "oversold_threshold": (20, 35),
            "overbought_threshold": (65, 85),
        }
        
        # Run random search
        result = await optimizer.random_search(
            strategy_code="rsi_mean_reversion",
            symbols=["AAPL"],
            start_date=date(2024, 1, 1),
            end_date=date(2024, 3, 31),
            parameter_bounds=parameter_bounds,
            n_iterations=20,
        )
        
        return result

# Run random search
random_result = await random_search_optimization()
print(random_result.summary())

## Running Backtest with Best Parameters

Run a full backtest using the optimal parameters found.

In [ ]:
async def run_optimized_backtest():
    # Get best parameters from optimization
    best_params = opt_result.best_parameters
    
    print(f"Running backtest with optimized parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    
    async with BacktestAdapter() as adapter:
        result = await adapter.run_backtest(
            strategy_code="rsi_mean_reversion",
            symbols=["AAPL", "0700.HK"],
            start_date=date(2024, 1, 1),
            end_date=date(2024, 12, 31),
            parameters=best_params,
        )
        
        return result

# Run optimized backtest
optimized_result = await run_optimized_backtest()
print("\nOptimized Backtest Results:")
print(optimized_result.summary())

## Comparing Results

Compare the default and optimized strategies.

In [ ]:
# Create comparison DataFrame
comparison_data = {
    "Metric": ["Total Return", "Sharpe Ratio", "Max Drawdown", "Win Rate", "Profit Factor"],
    "Default": [
        f"{result.metrics.total_return:.2f}%",
        f"{result.metrics.sharpe_ratio:.2f}",
        f"{result.metrics.max_drawdown:.2f}%",
        f"{result.metrics.win_rate:.1f}%",
        f"{result.metrics.profit_factor:.2f}",
    ],
    "Optimized": [
        f"{optimized_result.metrics.total_return:.2f}%",
        f"{optimized_result.metrics.sharpe_ratio:.2f}",
        f"{optimized_result.metrics.max_drawdown:.2f}%",
        f"{optimized_result.metrics.win_rate:.1f}%",
        f"{optimized_result.metrics.profit_factor:.2f}",
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

## Rolling Metrics Analysis

Calculate and plot rolling performance metrics.

In [ ]:
# Calculate rolling metrics
rolling = optimized_result.calculate_rolling_metrics(window=20)

# Plot rolling metrics
fig = rolling[['returns', 'volatility', 'sharpe']].plot(
    figsize=(12, 6),
    title="Rolling Performance Metrics (20-day window)",
    ylabel="Value"
)
fig.legend()

## Summary

In this notebook, we demonstrated:

1. **Basic Backtesting**: Running a simple backtest with custom parameters
2. **Result Analysis**: Viewing metrics, trades, and equity curves
3. **Visualization**: Plotting equity curves, drawdowns, and monthly returns
4. **Progress Tracking**: Monitoring backtest execution progress
5. **Parameter Optimization**: Using grid search and random search to find optimal parameters
6. **Comparison**: Comparing default vs optimized strategy performance

The CBSC Strategy SDK provides a comprehensive toolkit for strategy development and backtesting in Jupyter notebooks.